In [1]:
# Cell 1 — check GPU
!nvidia-smi

Fri Oct  2 06:49:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2 — install pinned deps (keep Kaggle's CUDA torch build; the actual torch version is recorded in config.json)
import glob, os
IN = os.path.dirname(glob.glob("/kaggle/input/**/sup_pairs.jsonl", recursive=True)[0])
print("dataset dir:", IN)
!grep -v '^torch==' {IN}/requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

dataset dir: /kaggle/input/datasets/juliodeaquino/simcse-snli-sup-inputs
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 10.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 89.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 61.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 104.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 85.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 88.0 MB/s eta 0:00:00:00:01
ERR

In [6]:
# Cell 3 — recreate the repo layout under /kaggle/working so paths resolve unmodified
import shutil
W = "/kaggle/working"
layout = {"train_simcse.py": "src/train_simcse.py", "evaluate.py": "src/evaluate.py",
          "supervised.json": "configs/supervised.json",
          "sup_pairs.jsonl": "data/processed/sup_pairs.jsonl",
          "stsb_dev.jsonl": "data/stsb/dev.jsonl", "requirements.txt": "requirements.txt"}
for src, dst in layout.items():
    os.makedirs(os.path.join(W, os.path.dirname(dst)), exist_ok=True)
    shutil.copy(os.path.join(IN, src), os.path.join(W, dst))
os.chdir(W)
!cd {IN} && sha256sum -c SHA256SUMS
!find src configs data -type f | sort

evaluate.py: OK
requirements.txt: OK
stsb_dev.jsonl: OK
supervised.json: OK
sup_pairs.jsonl: OK
train_simcse.py: OK
configs/supervised.json
data/processed/sup_pairs.jsonl
data/stsb/dev.jsonl
src/evaluate.py
src/train_simcse.py


In [7]:
# Cell 4 — train (Mode B, supervised with hard negatives, Eq. 5)
RUN_ID = "sup_seed42_bs128"
HW = f"Kaggle {torch.cuda.get_device_name(0).replace('Tesla ', '')} x{torch.cuda.device_count()}"
print(HW)
!cd /kaggle/working && python src/train_simcse.py --mode sup --config configs/supervised.json --output_dir runs/{RUN_ID} --hardware "{HW}"

Kaggle T4 x2
config.json: 100%|█████████████████████████████| 570/570 [00:00<00:00, 2.74MB/s]
tokenizer_config.json: 100%|██████████████████| 48.0/48.0 [00:00<00:00, 207kB/s]
vocab.txt: 100%|█████████████████████████████| 232k/232k [00:00<00:00, 7.50MB/s]
tokenizer.json: 100%|████████████████████████| 466k/466k [00:00<00:00, 11.5MB/s]

model.safetensors: downloading bytes: █████████▏            |  183MB, 11.2MB/s  
model.safetensors: downloading bytes: ██████████████████▊   |  376MB, 30.6MB/s  
model.safetensors: downloading bytes: ██████████████████████|  415MB, 38.8MB/s  
model.safetensors: reconstructing file: 100%|██████|  440MB /  440MB, 42.0MB/s  
Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 2762.03it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predic

In [8]:
# Cell 5 — verify run_record.json (written by the script) and trim the output
import json
rec = json.load(open(f"/kaggle/working/runs/{RUN_ID}/run_record.json"))
need = {"run_id","mode","timestamp","config","seed","hardware","results","checkpoint_path","notes"}
assert need <= rec.keys(), need - rec.keys()
assert rec["mode"] == "supervised" and rec["config"]["use_hard_negatives"] is True
assert rec["results"]["dev_spearman"] is not None and rec["results"]["test_spearman"] is None
print(json.dumps({k: rec[k] for k in ("run_id","hardware","results","notes")}, indent=2))
shutil.rmtree("/kaggle/working/data")   # inputs, no need to re-download them
!find /kaggle/working/runs -type f | sort && du -sh /kaggle/working/runs

{
  "run_id": "sup_seed42_bs128",
  "hardware": {
    "device": "Kaggle T4 x2",
    "num_gpus": 1,
    "num_gpus_available": 2,
    "precision": "fp16 (amp)",
    "gpu_name": "Tesla T4"
  },
  "results": {
    "dev_spearman": 0.8204571129313513,
    "test_spearman": null,
    "best_step": 350
  },
  "notes": ""
}
/kaggle/working/runs/sup_seed42_bs128/checkpoint/config.json
/kaggle/working/runs/sup_seed42_bs128/checkpoint/model.safetensors
/kaggle/working/runs/sup_seed42_bs128/checkpoint/tokenizer_config.json
/kaggle/working/runs/sup_seed42_bs128/checkpoint/tokenizer.json
/kaggle/working/runs/sup_seed42_bs128/config.json
/kaggle/working/runs/sup_seed42_bs128/results.json
/kaggle/working/runs/sup_seed42_bs128/run_record.json
419M	/kaggle/working/runs
